# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [41]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestRegressor,AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier 
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV



from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

SEED = 1

In [42]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


#### Preparing the data

In order to do that, first I will perform an EDA (same steps as previous lab with this dataset)

In [43]:
# Info about the dataset
spaceship.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


In [44]:
#Check for null values
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [45]:
spaceship.dropna(inplace=True)

In [46]:
# Cabin is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

spaceship["Cabin"] = spaceship["Cabin"].apply(lambda x: x.split("/")[0])
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [47]:
# Drop PassengerId and Name
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

**Perform Train Test Split**

In [48]:
# Select the features (X) and the target (y)
X = spaceship[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP','RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']]
y = spaceship['Transported']

# ============================================================
# Train / Test Split
# ============================================================

# Split the data into training (80%) and testing (20%)
# stratify=y keeps the same proportion of the target classes
# in both the training and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (5284, 11)
X_test shape: (1322, 11)


In [49]:
# ============================================================
# Separate Categorical and Numerical Features
# ============================================================

# Categorical variables that need One-Hot Encoding
categorical_cols = [
    'HomePlanet',
    'CryoSleep',
    'Cabin',
    'Destination',
    'VIP'
]

# Numerical variables that will be kept as numbers
numeric_cols = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

In [50]:
# ============================================================
# One-Hot Encoding
# ============================================================

# Create the encoder
# drop='first' avoids multicollinearity
# handle_unknown='ignore' prevents errors if new categories
# appear in the test set

encoder = OneHotEncoder(
    sparse_output=False,
    drop='first',
    handle_unknown='ignore'
)

# Learn the categories from the training data and transform it
X_train_cat = encoder.fit_transform(X_train[categorical_cols])

# Apply the same transformation to the test data
X_test_cat = encoder.transform(X_test[categorical_cols])

# Retrieve the names of the newly created columns
encoded_cols = encoder.get_feature_names_out(categorical_cols)

# Convert the encoded arrays back into DataFrames
# (this keeps the column names and original row indexes)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_cols,
    index=X_test.index
)


In [51]:
# ============================================================
# Keep the Numerical Features
# ============================================================

# Extract the numerical columns from the original dataset

X_train_num = X_train[numeric_cols]
X_test_num = X_test[numeric_cols]


# ============================================================
# Combine Numerical and Encoded Features
# ============================================================

# Join the numerical variables with the encoded categorical ones

X_train_final = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

In [52]:
# ============================================================
# Feature Scaling
# ============================================================

scaler = StandardScaler()

# Return a pandas DataFrame instead of a NumPy array
scaler.set_output(transform="pandas")

# Compute the mean and standard deviation using ONLY the training set
# and standardize the training data

X_train_final = scaler.fit_transform(X_train_final)

# Apply the same scaling to the test data

X_test_final = scaler.transform(X_test_final)


# ============================================================
# Check the Result
# ============================================================

print(X_train_final.head())
print(X_train_final.shape)
print(X_test_final.shape)

           Age  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck  \
6608 -0.062685    -0.347862  -0.194775     -0.298866  1.109513  1.010481   
7683  0.696693    -0.318144   1.929981     -0.298866  0.101711  0.197504   
8083 -0.545925    -0.347862  -0.293514     -0.298866 -0.278132 -0.264883   
4011  1.801242    -0.347862  -0.137999     -0.297178 -0.278132  0.229701   
6539 -0.269788    -0.347862   0.587738     -0.298866  1.059890 -0.264883   

      HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True   Cabin_B   Cabin_C  \
6608           1.704522        -0.513627       -0.742856 -0.328263  3.192948   
7683           1.704522        -0.513627       -0.742856 -0.328263  3.192948   
8083          -0.586675         1.946939        1.346155 -0.328263 -0.313190   
4011          -0.586675        -0.513627       -0.742856 -0.328263 -0.313190   
6539          -0.586675        -0.513627       -0.742856 -0.328263 -0.313190   

       Cabin_D   Cabin_E   Cabin_F  Cabin_G   Cabin_T  \
6608 

### Model trainings

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [53]:
#your code here

xgb_clf = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=2,
    subsample=0.9,
    colsample_bytree=1.0,
    gamma=0.5,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=SEED,
    eval_metric="logloss"
)

# Train the model
xgb_clf.fit(X_train_final, y_train)

print("Model Trained!")

Model Trained!


- Evaluate your model

In [66]:
# ============================================================
# Make Predictions
# ============================================================

y_pred_train = xgb_clf.predict(X_train_final)
y_pred_test = xgb_clf.predict(X_test_final)

# ============================================================
# Evaluate the Model
# ============================================================

accuracy_train = accuracy_score(y_train, y_pred_train)
print(f"Training Accuracy: {accuracy_train * 100:.2f}%")

accuracy_test = accuracy_score(y_test, y_pred_test)
print(f"Testing Accuracy: {accuracy_test * 100:.2f}%")

# Save baseline accuracies
baseline_train_accuracy = accuracy_train
baseline_test_accuracy = accuracy_test

print()

print(classification_report(
    y_test,
    y_pred_test,
    target_names=["Not Transported", "Transported"]
))

Training Accuracy: 84.33%
Testing Accuracy: 78.97%

                 precision    recall  f1-score   support

Not Transported       0.80      0.77      0.78       656
    Transported       0.78      0.81      0.80       666

       accuracy                           0.79      1322
      macro avg       0.79      0.79      0.79      1322
   weighted avg       0.79      0.79      0.79      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [60]:
param_grid = {
    "n_estimators": [200, 300],
    "learning_rate": [0.03, 0.1],
    "max_depth": [3, 5],
    "min_child_weight": [1, 3],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.5],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [1, 10]
}

- Run Grid Search

In [63]:
grid = GridSearchCV(
    estimator=XGBClassifier(
        random_state=SEED,
        eval_metric="logloss"
    ),
    param_grid=param_grid,
    cv=4,
    scoring="accuracy",
    n_jobs=-1,
)

grid.fit(X_train_final, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'colsample_bytree': 1.0, 'gamma': 0, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 300, 'reg_alpha': 0.1, 'reg_lambda': 10, 'subsample': 1.0}
Best CV Score: 0.8120741862225587


- Evaluate your model

In [67]:
best_xgb = grid.best_estimator_

y_pred_train = best_xgb.predict(X_train_final)
y_pred_test = best_xgb.predict(X_test_final)

print(f"Training Accuracy: {accuracy_score(y_train, y_pred_train)*100:.2f}%")
print(f"Testing Accuracy: {accuracy_score(y_test, y_pred_test)*100:.2f}%")

accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred_test)

print(classification_report(
    y_test,
    y_pred_test,
    target_names=["Not Transported", "Transported"]
))

Training Accuracy: 82.89%
Testing Accuracy: 79.27%
                 precision    recall  f1-score   support

Not Transported       0.81      0.77      0.79       656
    Transported       0.78      0.82      0.80       666

       accuracy                           0.79      1322
      macro avg       0.79      0.79      0.79      1322
   weighted avg       0.79      0.79      0.79      1322



Let's compare them

In [70]:
print("\n=== Comparison ===")
print(f"Training Accuracy: {baseline_train_accuracy:.2%} → {accuracy_train:.2%} ({accuracy_train - baseline_train_accuracy:+.2%})")
print(f"Testing Accuracy : {baseline_test_accuracy:.2%} → {accuracy_test:.2%} ({accuracy_test - baseline_test_accuracy:+.2%})")

if accuracy_test > baseline_test_accuracy:
    print("\n✅ Grid Search improved the model's generalization.")
else:
    print("\n⚠️ Grid Search did not improve the model's generalization.")


=== Comparison ===
Training Accuracy: 84.33% → 82.89% (-1.44%)
Testing Accuracy : 78.97% → 79.27% (+0.30%)

✅ Grid Search improved the model's generalization.


## Conclusion

After applying Grid Search, the XGBoost classifier achieved a 0.30% improvement in testing accuracy, while the training accuracy decreased by 1.44%. This indicates that the tuned model generalizes better to unseen data and is less prone to overfitting than the baseline model.